# Classical CV pipeline - area selection & scale calibration

Helper notebook used to (1) measure the dish ROI on the first frame of an
isolated-dish video and (2) determine the mm/px scale from a reference object of
known size. Run interactively (OpenCV windows): select the ROI, then press 'q'.

Input: `../records/<dataset>/isolated_dishes/<condition>_dish_<larva>.mp4`


## 1. Imports

In [ ]:
import cv2 as cv
import sys

In [ ]:
# Maps a condition number to a file name.
STUDY_TYPES = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x10_7",
    5: "coli_5x10_8",
}

## 2. Mark a ROI on the first frame to measure dimensions

In [ ]:
def display_study_types():
    """Print the available study conditions."""
    print("\n=== Available conditions ===")
    for num, name in STUDY_TYPES.items():
        print(f"{num}. {name}")
    print("============================\n")

def get_study_type():
    """Read the condition number from the user."""
    display_study_types()
    while True:
        try:
            study_num = int(input('Enter condition number: '))
            if study_num in STUDY_TYPES:
                return STUDY_TYPES[study_num]
            else:
                print(f'Error: choose a number between 1 and {len(STUDY_TYPES)}')
        except ValueError:
            print('Error: enter an integer')

def get_larva_number():
    """Read the larva number from the user."""
    while True:
        try:
            larva_num = int(input('Enter larva number (1-6): '))
            if 1 <= larva_num <= 6:
                return larva_num
            else:
                print('Error: larva number must be between 1 and 6')
        except ValueError:
            print('Error: enter an integer')

def open_video_capture(study_type, larva_number, dataset="session1"):
    """Open the video for the given condition and larva number."""
    video_path = f'../records/{dataset}/isolated_dishes/{study_type}_dish_{larva_number}.mp4'
    capture = cv.VideoCapture(video_path)
    if not capture.isOpened():
        print(f'Error: cannot open video: {video_path}')
        sys.exit()
    return capture

def read_first_frame(capture):
    """Read the first frame of the video."""
    ret, frame = capture.read()
    if not ret:
        print('Error: cannot read the first frame of the video.')
        sys.exit()
    return frame

def select_roi(frame):
    """Let the user select a region of interest (ROI) on the frame."""
    bbox = cv.selectROI(frame)
    return bbox

def display_result(frame, bbox):
    """Show the frame with the marked ROI and its coordinates."""
    x, y, w, h = bbox
    cv.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
    roi_text = f'ROI: (x: {x}, y: {y}, w: {w}, h: {h})'
    cv.putText(frame, roi_text, (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv.imshow('Frame with ROI', frame)

def main():
    """Main program function."""
    study_type = get_study_type()
    larva_number = get_larva_number()
    capture = open_video_capture(study_type, larva_number)
    frame = read_first_frame(capture)

    bbox = select_roi(frame)
    display_result(frame, bbox)

    print("\nPress 'q' in the window to close and continue...")
    while True:
        key = cv.waitKey(1) & 0xFF
        if key == ord('q'):
            break

    cv.destroyAllWindows()
    cv.waitKey(1)
    capture.release()

if __name__ == "__main__":
    main()

## 3. Determine the scale (mm/px)

In [ ]:
def get_real_dimension():
    """Read the real size of the reference object from the user."""
    return float(input('Enter the real size of the reference object (mm): '))

def calculate_scale(real_dimension_mm, pixel_dimension):
    """Compute the scale from the real and pixel widths."""
    return real_dimension_mm / pixel_dimension

def add_scale_text(frame, scale):
    """Add the scale info as text on the image."""
    font = cv.FONT_HERSHEY_SIMPLEX
    text = f'Scale: {scale:.3f} mm/px'
    cv.putText(frame, text, (10, 30), font, 1, (255, 255, 255), 2, cv.LINE_AA)

def main_scale():
    """Main scale-calibration function."""
    study_type = get_study_type()
    larva_number = get_larva_number()
    real_dimension_mm = get_real_dimension()
    capture = open_video_capture(study_type, larva_number)
    frame = read_first_frame(capture)

    # Select the reference area
    bbox = cv.selectROI(frame)
    pixel_dimension = bbox[2]  # side width in pixels (square reference)

    scale = calculate_scale(real_dimension_mm, pixel_dimension)
    print(f'Scale: {scale:.3f} mm per pixel')

    add_scale_text(frame, scale)
    cv.imshow("ROI and Scale", frame)

    print("\nPress 'q' in the window to close and continue...")
    while True:
        key = cv.waitKey(1) & 0xFF
        if key == ord('q'):
            break

    cv.destroyAllWindows()
    cv.waitKey(1)
    capture.release()

if __name__ == "__main__":
    main_scale()